[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aai-services/datainsight-curriculum/blob/main/stages/stage-1/1.5-describing-data/lesson.ipynb)

# 1.5 Describing data

Week 8.

By the end of this lesson you can:

- Describe a distribution by its shape, center, and spread
- Choose between the mean and the median, and between the standard deviation and the interquartile range
- Compare groups with charts that show the whole distribution, not just an average

We use GDP per person in 2020 (World Bank, via Our World in Data, CC BY 4.0) and the Palmer Penguins (CC0).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = "https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/datasets/"

countries = pd.read_csv(DATA + "countries.csv")
penguins = pd.read_csv(DATA + "penguins.csv")

gdp = countries.loc[countries["year"] == 2020, "gdp_per_capita"].dropna()
print(len(gdp), "countries with GDP per person in 2020")

## Shape first

Before computing any average, look at the distribution. A histogram counts how many values fall in each range.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(gdp, bins=30, color="#4F9FDF", edgecolor="white")
ax.set_title("GDP per person, 2020")
ax.set_xlabel("GDP per person (international $, PPP)")
ax.set_ylabel("Number of countries")
plt.show()

The distribution is **right-skewed**: most countries are bunched at lower values, with a long tail of a few very rich countries.

## Center: mean or median?

In [ ]:
print(f"Mean:   {gdp.mean():,.0f}")
print(f"Median: {gdp.median():,.0f}")

The long right tail pulls the **mean** well above the **median**. The median, the middle value, better describes a typical country here. Use the median for skewed data, and the mean for roughly symmetric data or when the total matters.

## Spread: standard deviation or interquartile range?

The **standard deviation** (SD) measures typical distance from the mean, and is sensitive to extreme values like the mean is. The **interquartile range** (IQR) is the range of the middle half of the data, from the 25th to the 75th percentile, and is robust to extremes.

In [ ]:
q1, q3 = gdp.quantile([0.25, 0.75])
print(f"SD:  {gdp.std():,.0f}")
print(f"IQR: {q3 - q1:,.0f}  (from {q1:,.0f} to {q3:,.0f})")

Pair them: **median with IQR** for skewed data, **mean with SD** for roughly symmetric data.

## A log scale for skewed data

On a log scale, each step is a multiplication (1,000, then 10,000, then 100,000). The shape of the skewed data becomes much easier to read.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(np.log10(gdp), bins=30, color="#4F9FDF", edgecolor="white")
ax.set_title("GDP per person, 2020 (log scale)")
ax.set_xlabel("GDP per person (international $, PPP)")
ax.set_ylabel("Number of countries")
ticks = [3, 4, 5]
ax.set_xticks(ticks, [f"{10**t:,}" for t in ticks])
plt.show()

## Comparing groups

To compare groups, show the whole distribution for each group. A box plot shows the median (the line), the IQR (the box), and points far from the rest.

In [ ]:
species = ["Adelie", "Chinstrap", "Gentoo"]
groups = [penguins.loc[penguins["species"] == s, "flipper_length_mm"].dropna() for s in species]

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(groups)
ax.set_xticks([1, 2, 3], species)
for i, values in enumerate(groups, start=1):   # show every penguin, slightly spread out
    ax.scatter(np.random.default_rng(i).normal(i, 0.05, len(values)), values, s=6, alpha=0.4)
ax.set_title("Flipper length by species")
ax.set_ylabel("Flipper length (mm)")
plt.show()

In [ ]:
penguins.groupby("species")["flipper_length_mm"].agg(["count", "mean", "median", "std"]).round(1)

## Worked example: describing life expectancy by region

A good description reports shape, center, spread, and count, and names what is missing.

In [ ]:
life_2020 = countries[(countries["year"] == 2020)].dropna(subset=["life_expectancy", "region"])
summary = life_2020.groupby("region")["life_expectancy"].agg(
    countries="count",
    median="median",
    q1=lambda s: s.quantile(0.25),
    q3=lambda s: s.quantile(0.75),
)
summary.round(1)

**A written description:**

> In 2020, median life expectancy was lowest in Africa and highest in Europe. The spread within regions is large: the middle half of African countries spans several years, so a single regional number hides a lot of variation. The table covers countries with both a region and a 2020 value; territories without a region classification are not included.

Notice the description names the measure, the spread, and what is left out. That is the standard for your projects.

## Your turn

Open `practice-1.5.ipynb`.